# Project 24: Mobile App Usage & User Engagement Analysis
**Industry:** Technology / Mobile Applications  
**Goal:** Study app sessions, features used, session duration and user ratings to understand engagement and improve user experience.

> Edit the `COLS` dictionary in the first code cell if your CSV uses different column names.

In [1]:
import pandas as pd
import numpy as np

# ---- EDIT THESE IF YOUR COLUMN NAMES DIFFER ----
COLS = {
    "session": "Session_ID",
    "user": "User_ID",
    "date": "Session_Date",
    "duration": "Session_Duration_Mins",
    "rating": "User_Rating",
    "feature": "Feature_Used",
    "device": "Device_Type",
    "age": "Age_Group",
}
C = COLS

## Task 1: Load and inspect the dataset

In [2]:
df = pd.read_csv("24_mobile_app_usage.csv")
print("Shape:", df.shape)
df.head()

Shape: (1008, 8)


,Session_ID,User_ID,Session_Date,Session_Duration_Mins,User_Rating,Feature_Used,Device_Type,Age_Group
0,S00084,U0237,2026-10-02 14:14:00,4.4,4.0,Stories,iOS,18-24
1,S00677,U0167,2026-07-11 15:21:00,7.5,3.0,Payments,Android,45-54
2,S00832,U0033,2026-03-21 16:12:00,5.3,NaN,Video Player,Android,35-44
3,S00894,U0015,2026-11-20 05:12:00,10.2,2.0,Shopping Cart,Android,18-24
4,S00848,U0030,2026-01-18 13:21:00,13.6,5.0,Home Feed,iOS,35-44


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1008 entries, 0 to 1007
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Session_ID             1008 non-null   str    
 1   User_ID                1008 non-null   str    
 2   Session_Date           1008 non-null   str    
 3   Session_Duration_Mins  968 non-null    float64
 4   User_Rating            953 non-null    float64
 5   Feature_Used           1008 non-null   str    
 6   Device_Type            1008 non-null   str    
 7   Age_Group              1008 non-null   str    
dtypes: float64(2), str(6)
memory usage: 63.1 KB


In [4]:
df.describe(include="all")

,Session_ID,User_ID,Session_Date,Session_Duration_Mins,User_Rating,Feature_Used,Device_Type,Age_Group
count,1008,1008,1008,968.000000,953.000000,1008,1008,1008
unique,1000,288,998,NaN,NaN,12,3,5
top,S00369,U0021,2026-01-08 07:25:00,NaN,NaN,Home Feed,Android,18-24
freq,2,10,2,NaN,NaN,185,512,284
mean,NaN,NaN,NaN,13.128099,3.693599,NaN,NaN,NaN
std,NaN,NaN,NaN,13.562848,1.164302,NaN,NaN,NaN
min,NaN,NaN,NaN,1.100000,1.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,6.600000,3.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,10.500000,4.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,15.425000,5.000000,NaN,NaN,NaN


## Task 2: Check missing values and duplicate Session_ID records

In [5]:
print("Missing values per column:")
print(df.isna().sum())
dup_count = df.duplicated(subset=C["session"]).sum()
print("\nDuplicate Session_ID records:", dup_count)

Missing values per column:
Session_ID                0
User_ID                   0
Session_Date              0
Session_Duration_Mins    40
User_Rating              55
Feature_Used              0
Device_Type               0
Age_Group                 0
dtype: int64

Duplicate Session_ID records: 8


In [6]:
# Keep the first occurrence of any duplicated Session_ID
df = df.drop_duplicates(subset=C["session"], keep="first")
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (1000, 8)


## Task 3: Handle missing Session_Duration_Mins / User_Rating
**Approach and justification:**
- Session durations are usually right-skewed with outliers, so the **median** is more robust than the mean.
- Ratings are ordinal (e.g. 1-5), so the **median** keeps imputed values valid.
- Dropping rows would throw away useful information (feature, device, date) from those sessions.

In [7]:
df[C["duration"]] = df[C["duration"]].fillna(df[C["duration"]].median())
df[C["rating"]] = df[C["rating"]].fillna(df[C["rating"]].median())
df[[C["duration"], C["rating"]]].isna().sum()

Session_Duration_Mins    0
User_Rating              0
dtype: int64

## Task 4: Convert Session_Date to datetime; extract month, weekday, hour

In [8]:
df[C["date"]] = pd.to_datetime(df[C["date"]], errors="coerce")
df["Month"] = df[C["date"]].dt.month_name()
df["Weekday"] = df[C["date"]].dt.day_name()
df["Hour"] = df[C["date"]].dt.hour
df[[C["date"], "Month", "Weekday", "Hour"]].head()

,Session_Date,Month,Weekday,Hour
0,2026-10-02 14:14:00,October,Friday,14
1,2026-07-11 15:21:00,July,Saturday,15
2,2026-03-21 16:12:00,March,Saturday,16
3,2026-11-20 05:12:00,November,Friday,5
4,2026-01-18 13:21:00,January,Sunday,13


## Task 5: Total sessions, unique users, average session duration

In [9]:
total_sessions = len(df)
unique_users = df[C["user"]].nunique()
avg_duration = df[C["duration"]].mean()
print(f"Total sessions: {total_sessions}")
print(f"Unique users: {unique_users}")
print(f"Average session duration: {avg_duration:.2f} mins")

Total sessions: 1000
Unique users: 288
Average session duration: 13.07 mins


## Task 6: 10 most frequently used app features

In [10]:
feature_popularity = df[C["feature"]].value_counts().head(10)
feature_popularity.to_frame("Sessions")

,Sessions
Feature_Used,
Home Feed,183
Search,161
Messaging,147
Notifications,112
Video Player,92
Profile,79
Shopping Cart,56
Settings,49
Payments,41


## Task 7: Average session duration across device types

In [11]:
device_comparison = (
    df.groupby(C["device"])[C["duration"]]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
)
device_comparison

,mean,median,count
Device_Type,,,
Tablet,13.123016,10.0,126
Android,13.083922,10.5,510
iOS,13.019505,10.5,364


## Task 8: Rank user age groups by total sessions

In [12]:
age_rank = df[C["age"]].value_counts().rename("Sessions").to_frame()
age_rank["Rank"] = age_rank["Sessions"].rank(ascending=False, method="min").astype(int)
age_rank

,Sessions,Rank
Age_Group,,
18-24,281,1
25-34,275,2
35-44,201,3
45-54,170,4
55+,73,5


## Task 9: Average user ratings across app features

In [13]:
rating_by_feature = (
    df.groupby(C["feature"])[C["rating"]].mean().sort_values(ascending=False)
)
rating_by_feature.to_frame("Avg Rating")

,Avg Rating
Feature_Used,
Wishlist,4.052632
Stories,3.861111
Messaging,3.775510
Home Feed,3.732240
Search,3.720497
Notifications,3.705357
Shopping Cart,3.696429
Payments,3.682927
Video Player,3.652174


## Task 10: Peak usage hours and busiest weekdays

In [14]:
peak_hours = df["Hour"].value_counts().head(5)
print("Top 5 peak hours:")
print(peak_hours)

Top 5 peak hours:
Hour
20    89
18    84
19    78
21    72
17    67
Name: count, dtype: int64


## Task 11: List for most-used features, tuple for fixed device categories

In [15]:
top_features_list = feature_popularity.index.tolist()          # list (mutable)
device_categories = tuple(sorted(df[C["device"]].unique()))    # tuple (fixed)
print("Most-used features (list):", top_features_list)
print("Device categories (tuple):", device_categories)

Most-used features (list): ['Home Feed', 'Search', 'Messaging', 'Notifications', 'Video Player', 'Profile', 'Shopping Cart', 'Settings', 'Payments', 'Stories']
Device categories (tuple): ('Android', 'Tablet', 'iOS')


## Task 12: Set for unique users + function for average duration of a feature

In [16]:
unique_user_set = set(df[C["user"]])
print("Unique users via set:", len(unique_user_set))

def avg_duration_for_feature(data, feature_name):
    """Return the average session duration for a selected feature."""
    subset = data[data[C["feature"]] == feature_name]
    if subset.empty:
        return None
    return subset[C["duration"]].mean()

example_feature = top_features_list[0]
print(f"Avg duration for '{example_feature}':",
      round(avg_duration_for_feature(df, example_feature), 2), "mins")

Unique users via set: 288
Avg duration for 'Home Feed': 14.14 mins


## Task 13: NumPy mean, median, standard deviation of session duration

In [17]:
durations = df[C["duration"]].to_numpy()
print("Mean:  ", np.mean(durations))
print("Median:", np.median(durations))
print("Std:   ", np.std(durations))

Mean:   13.065400000000002
Median: 10.5
Std:    13.335617077585873


## Task 14: Outliers (IQR method) and engagement labels
**Outlier rule:** a session is unusually long if duration > Q3 + 1.5 x IQR.

**Engagement label rules (documented):**
- **Low:** duration < Q1
- **Medium:** Q1 <= duration <= Q3
- **High:** duration > Q3

In [18]:
q1, q3 = np.percentile(durations, [25, 75])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
df["Long_Session_Outlier"] = df[C["duration"]] > upper_fence
print(f"Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, upper fence={upper_fence:.2f}")
print("Unusually long sessions:", df["Long_Session_Outlier"].sum())

Q1=6.70, Q3=15.20, IQR=8.50, upper fence=27.95
Unusually long sessions: 48


In [19]:
def engagement_label(minutes):
    if minutes < q1:
        return "Low"
    elif minutes <= q3:
        return "Medium"
    return "High"

df["Engagement"] = df[C["duration"]].apply(engagement_label)
df["Engagement"].value_counts()

Engagement
Medium    506
High      248
Low       246
Name: count, dtype: int64

## Task 15: AppUser class and five recommendations

In [20]:
class AppUser:
    def __init__(self, user_id, device_type):
        self.user_id = user_id
        self.device_type = device_type

    def __repr__(self):
        return f"AppUser(user_id={self.user_id}, device_type={self.device_type})"

first = df.iloc[0]
sample_user = AppUser(first[C["user"]], first[C["device"]])
sample_user

AppUser(user_id=U0237, device_type=iOS)

### Five recommendations to improve engagement
> **Replace the details below with YOUR actual results** (feature names, hours, devices, age groups).

1. **Promote high-rated, low-usage features** - users who try them like them, so surface them in onboarding and the home screen.
2. **Improve the lowest-rated features** - redesign or fix pain points in features with the lowest average ratings.
3. **Time notifications to peak hours and busiest weekdays** - send pushes and offers when users are most active.
4. **Optimize for the device type with the shortest sessions** - check performance and layout issues there.
5. **Target the least-active age group** - tailor content, onboarding and incentives to raise their session counts.

## Summary of Findings
*Write 4-6 bullet points here using your real numbers: total sessions, unique users, top feature, best/worst device, peak hour, busiest day, share of High-engagement sessions.*